# Article Analysis

Este notebook lê apenas os artefatos salvos pelos scripts de treino e gera tabelas e gráficos para o trabalho.

In [ ]:
from pathlib import Path
import json
import torch
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'src').exists() and (REPO_ROOT.parent / 'src').exists():
    REPO_ROOT = REPO_ROOT.parent

RUN_ROOT = REPO_ROOT / 'outputs' / 'cla_lora_runs'
if not RUN_ROOT.exists() and (REPO_ROOT / 'cla_lora_runs').exists():
    RUN_ROOT = REPO_ROOT / 'cla_lora_runs'

SELECTED_RUNS = {
    'full_finetune': '',
    'lora': '',
    'adalora': '',
}

K = 4


In [ ]:
def load_json(path):
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def load_run(method):
    run_name = SELECTED_RUNS[method]
    if not run_name:
        return None

    run_dir = RUN_ROOT / method / run_name
    if not run_dir.exists():
        return None

    config = load_json(run_dir / 'config.json')
    summary = load_json(run_dir / 'summary.json')
    history = pd.read_csv(run_dir / 'train_history.csv')
    return {
        'run_dir': run_dir,
        'config': config,
        'summary': summary,
        'history': history,
        'deltas': torch.load(run_dir / 'target_deltas.pt', map_location='cpu'),
        'svdvals': torch.load(run_dir / 'target_svdvals.pt', map_location='cpu'),
        'layer_stats': load_json(run_dir / 'layer_stats.json'),
        'display_name': config.get('display_name', method),
    }

def principal_subspace(delta, side, k):
    u, s, vh = torch.linalg.svd(delta.float(), full_matrices=False)
    if s.numel() == 0:
        return None
    rank = min(k, s.numel())
    if side == 'left':
        return u[:, :rank]
    return vh[:rank, :].T

def subspace_similarity(delta_ref, delta_cmp, k):
    metrics = {}
    for side in ['left', 'right']:
        basis_ref = principal_subspace(delta_ref, side, k)
        basis_cmp = principal_subspace(delta_cmp, side, k)
        gram = basis_ref.T @ basis_cmp
        cosines = torch.linalg.svdvals(gram).clamp(0.0, 1.0)
        angles = torch.rad2deg(torch.acos(cosines))
        overlap = torch.linalg.matrix_norm(gram, ord='fro').item() ** 2 / gram.shape[0]
        metrics[f'{side}_mean_cosine'] = cosines.mean().item()
        metrics[f'{side}_max_angle_deg'] = angles.max().item()
        metrics[f'{side}_overlap'] = overlap
    return metrics


In [ ]:
runs = {}
load_issues = []

for method in SELECTED_RUNS:
    if not SELECTED_RUNS[method]:
        load_issues.append(f'Selecione um run para {method} em SELECTED_RUNS.')
        continue

    run = load_run(method)
    if run is None:
        load_issues.append(f'Run não encontrado para {method}: {SELECTED_RUNS[method]}')
        continue

    runs[method] = run

audit_rows = []
for method, run in runs.items():
    config = run['config']
    history = run['history']
    audit_rows.append({
        'method': run['display_name'],
        'run_dir': str(run['run_dir']),
        'comparison_suite_id': config.get('comparison_suite_id'),
        'loss_protocol': config.get('loss_protocol'),
        'max_length': config.get('max_length'),
        'seed': config.get('seed'),
        'learning_rate': config.get('learning_rate'),
        'effective_batch_size': config.get('effective_batch_size'),
        'num_epochs': config.get('num_epochs'),
        'epoch_minus_one_present': bool((history['epoch'] == -1).any()),
        'legacy_test_loss_column': 'legacy_test_loss' in history.columns,
    })

if audit_rows:
    display(pd.DataFrame(audit_rows))
else:
    print('Nenhum run carregado.')

if load_issues:
    print('Pendências de carregamento:')
    for issue in load_issues:
        print(f'- {issue}')


In [ ]:
summary_rows = []
for method, run in runs.items():
    summary = run['summary']
    summary_rows.append({
        'method': run['display_name'],
        'trainable_params': summary.get('trainable_params'),
        'trainable_pct': summary.get('trainable_pct'),
        'pretrain_legacy_test_loss': summary.get('pretrain_legacy_test_loss'),
        'final_legacy_test_loss': summary.get('final_legacy_test_loss'),
        'final_train_model_loss': summary.get('final_train_model_loss'),
        'final_train_objective_loss': summary.get('final_train_objective_loss'),
        'total_time_sec': summary.get('total_time_sec'),
    })

if summary_rows:
    display(pd.DataFrame(summary_rows))

print('Obs.: use train_objective_loss do AdaLoRA apenas como diagnóstico do próprio método, porque ela inclui regularização.')


In [ ]:
required_methods = ['full_finetune', 'lora', 'adalora']
comparison_issues = []

for method in required_methods:
    if method not in runs:
        comparison_issues.append(f'Método ausente na comparação principal: {method}')

if not comparison_issues:
    required_equal_fields = [
        'comparison_suite_id',
        'loss_protocol',
        'max_length',
        'seed',
        'learning_rate',
        'effective_batch_size',
        'num_epochs',
    ]

    for field in required_equal_fields:
        values = {runs[method]['config'].get(field) for method in required_methods}
        if None in values or len(values) != 1:
            comparison_issues.append(f'Campo incompatível para a curva principal: {field} -> {sorted(values, key=str)}')

    for method in required_methods:
        history = runs[method]['history']
        if 'legacy_test_loss' not in history.columns:
            comparison_issues.append(f'{method} não possui a coluna legacy_test_loss.')
        if not (history['epoch'] == -1).any():
            comparison_issues.append(f'{method} não possui a avaliação inicial com epoch = -1.')

if comparison_issues:
    print('Auditoria de compatibilidade:')
    for issue in comparison_issues:
        print(f'- {issue}')
else:
    print('Os runs selecionados estão compatíveis para a curva principal em legacy_test_loss.')


In [ ]:
if comparison_issues:
    print('A curva principal foi bloqueada pela auditoria de compatibilidade.')
else:
    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    for method in required_methods:
        history = runs[method]['history'].sort_values('epoch')
        ax.plot(
            history['epoch'],
            history['legacy_test_loss'],
            marker='o',
            label=runs[method]['display_name'],
        )
    ax.set_title('Legacy test loss por época')
    ax.set_xlabel('Epoca')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.tight_layout()


In [ ]:
masked_methods = [method for method in required_methods if method in runs and 'masked_test_loss' in runs[method]['history'].columns]
if masked_methods:
    fig, ax = plt.subplots(1, 1, figsize=(7, 4))
    for method in masked_methods:
        history = runs[method]['history'].sort_values('epoch')
        ax.plot(
            history['epoch'],
            history['masked_test_loss'],
            marker='o',
            label=runs[method]['display_name'],
        )
    ax.set_title('Masked test loss por época (diagnóstico)')
    ax.set_xlabel('Epoca')
    ax.set_ylabel('Loss')
    ax.legend()
    plt.tight_layout()
else:
    print('Nenhum run selecionado possui masked_test_loss. Isso é esperado se você estiver usando apenas o protocolo legacy_controlled_v1.')


In [ ]:
reference_method = next((method for method in ['full_finetune', 'lora', 'adalora'] if method in runs), None)
attention_layers = []

if reference_method is not None:
    attention_layers = sorted([
        name for name in runs[reference_method]['deltas'].keys()
        if '.attn.c_attn' in name
    ])

attention_layers[:3]


In [ ]:
if attention_layers:
    layer_name = attention_layers[0]
    plt.figure(figsize=(8, 4))
    for method, run in runs.items():
        svdvals = run['svdvals'][layer_name]
        plt.plot(range(1, len(svdvals) + 1), svdvals.numpy(), marker='o', label=run['display_name'])
    plt.yscale('log')
    plt.title(f'Espectro singular: {layer_name}')
    plt.xlabel('Índice')
    plt.ylabel('Valor singular')
    plt.legend()
    plt.tight_layout()
else:
    print('Nenhuma camada de atenção disponível para plotar.')


In [ ]:
svd_rows = []
for method, run in runs.items():
    for layer_name, stats in run['layer_stats'].items():
        if '.attn.c_attn' in layer_name:
            svd_rows.append({
                'method': run['display_name'],
                'layer': layer_name,
                'stable_rank': stats['stable_rank'],
                'energy_90_rank': stats['energy_90_rank'],
                'energy_95_rank': stats['energy_95_rank'],
            })
pd.DataFrame(svd_rows).head()


In [ ]:
similarity_rows = []
base_method = 'full_finetune' if 'full_finetune' in runs else ('lora' if 'lora' in runs else None)

if base_method is not None:
    comparison_methods = [method for method in ['lora', 'adalora'] if method in runs and method != base_method]
    for method in comparison_methods:
        for layer_name in attention_layers:
            metrics = subspace_similarity(
                runs[base_method]['deltas'][layer_name],
                runs[method]['deltas'][layer_name],
                K,
            )
            similarity_rows.append({
                'reference': runs[base_method]['display_name'],
                'method': runs[method]['display_name'],
                'layer': layer_name,
                **metrics,
            })

pd.DataFrame(similarity_rows).head()


In [ ]:
adalora_mlp_rows = []
if 'adalora' in runs:
    for layer_name, stats in runs['adalora']['layer_stats'].items():
        if '.mlp.' in layer_name:
            adalora_mlp_rows.append({'layer': layer_name, **stats})
pd.DataFrame(adalora_mlp_rows).head()
